# 04 — Spatial statistics

**Day 2, 10:15–11:15**

### Where we are going

So far we have done scRNA-seq with coordinates attached. Now we use the coordinates.

The organising idea: **build a graph of which cells touch which, then ask questions
of that graph.** Almost every method in the field is a variation on this.

By the end you can:

1. build and inspect a spatial neighbourhood graph, and justify its parameters
2. test which cell types sit next to which, with a proper permutation null
3. measure at what *distance scale* an interaction happens (co-occurrence, Ripley)
4. find genes with spatially structured expression (Moran's I) without using clusters
5. define **niches** — recurrent tissue neighbourhoods — and read them anatomically

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import squidpy as sq

sc.settings.verbosity = 1
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
NAVY, GOLD, CORAL, ICE = "#001158", "#FBAE40", "#F26B43", "#BCD2FF"

# This is your own annotation if you finished notebook 03, and the shipped one
# if you did not — either way day 2 runs. To force the shipped version, load
# ovarian_annotated_reference.h5ad instead.
adata = sc.read_h5ad(DATA / "ovarian_annotated.h5ad")

# Cell coordinates, in microns. Defined once here because almost every plot
# below needs them — keeping them in the setup cell means you can re-run any
# single plot later without hunting for where x and y came from.
x, y = adata.obsm["spatial"].T

print(adata)
print(f"\n{adata.n_obs:,} cells, x {x.min():.0f}-{x.max():.0f} um, "
      f"y {y.min():.0f}-{y.max():.0f} um")

In [ ]:
# ---- morphology image, georeferenced ------------------------------------- #
# The .h5ad covers the whole cell crop (~1.5 x 1.5 mm), but the image covers only
# the smaller inner window. To draw them together we need to know where that
# window sits in micron coordinates.
import json

def load_morphology(data_dir):
    """Return (image, extent_in_um) or (None, None) if the image is absent.

    extent is (x0, x1, y0, y1) in the same micron coordinates as
    adata.obsm["spatial"], so the image can be placed under the cells.
    """
    img_path = data_dir / "morphology_crop.ome.tif"
    if not img_path.exists():
        print("no morphology_crop.ome.tif — plots will fall back to a plain background")
        return None, None

    import tifffile
    img = tifffile.imread(img_path)

    # where is it? crop_metadata.json is authoritative; the transcript table is a
    # good fallback because it was cropped to exactly the same window
    meta_path = data_dir / "crop_metadata.json"
    if meta_path.exists():
        w = json.loads(meta_path.read_text())
        w = w.get("inner_window_um") or w.get("transcript_window_um")
        ext = (w["x0"], w["x0"] + w["width"], w["y0"], w["y0"] + w["height"])
    else:
        tx = pd.read_parquet(data_dir / "transcripts_crop.parquet",
                             columns=["x_location", "y_location"])
        ext = (tx["x_location"].min(), tx["x_location"].max(),
               tx["y_location"].min(), tx["y_location"].max())
        print("crop_metadata.json missing — inferring extent from the transcripts")

    ny, nx = img.shape[-2], img.shape[-1]
    px = (ext[1] - ext[0]) / nx
    print(f"image {img.shape}, extent x {ext[0]:.0f}-{ext[1]:.0f}, "
          f"y {ext[2]:.0f}-{ext[3]:.0f} um, {px:.3g} um/px")
    return img, ext


def show_morphology(ax, img, ext, channel=0, cmap="gray", vmax_pct=99.5, alpha=1.0):
    """Draw one channel with correct micron coordinates and image y-orientation."""
    plane = img[channel] if img.ndim == 3 else img
    x0, x1, y0, y1 = ext

    vmin, vmax = np.percentile(plane, 1), np.percentile(plane, vmax_pct)
    if vmax <= vmin:                    # near-uniform channel: fall back to range
        vmin, vmax = float(plane.min()), float(plane.max())
    if vmax <= vmin:
        vmax = vmin + 1                 # perfectly flat; avoid a blank panel

    ax.imshow(plane, cmap=cmap, alpha=alpha, origin="upper",
              extent=(x0, x1, y1, y0),          # note: bottom=y1, top=y0
              vmin=vmin, vmax=vmax,
              interpolation="bilinear", zorder=0)


img, img_ext = load_morphology(DATA)

## 1. The spatial graph — and why the choice is not innocent

Three ways to decide who is a neighbour, each encoding a different biological claim:

| Method | Claim | Use when |
|---|---|---|
| **k nearest neighbours** (`n_neighs=6`) | every cell has the same number of contacts | you want comparable statistics across regions |
| **Delaunay triangulation** (`delaunay=True`) | neighbours are cells sharing a Voronoi face — closest to "physically touching" | you care about direct contact |
| **Radius** (`radius=30`) | neighbours are anything within X um | you want a fixed physical scale, e.g. a signalling range |

Radius graphs behave badly where density varies — a cell in dense lymphoid tissue
gets 40 neighbours, one in loose stroma gets 2, and any per-cell statistic is then
confounded by density. kNN and Delaunay adapt. **Delaunay is usually the honest
default for "which cells are in contact".**

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="generic", delaunay=True, key_added="spatial")
conn = adata.obsp["spatial_connectivities"]
deg = np.asarray(conn.sum(axis=1)).ravel()
print(f"mean neighbours per cell: {deg.mean():.1f}   (Delaunay in 2D -> ~6)")

dist = adata.obsp["spatial_distances"]
d = dist.data
print(f"median edge length: {np.median(d):.1f} um   95th pct: {np.percentile(d, 95):.1f} um")

> **Try it yourself — who is a neighbour?**
>
> The three graph types from the table above give different answers. Build each and
> compare the average number of neighbours per cell. This does not overwrite the
> Delaunay graph the rest of the notebook uses.

In [ ]:
GRAPH = "delaunay"        # <-- CHANGE THIS: "delaunay", "knn", or "radius"

if GRAPH == "delaunay":
    sq.gr.spatial_neighbors(adata, coord_type="generic", delaunay=True,
                            key_added="try")
elif GRAPH == "knn":
    sq.gr.spatial_neighbors(adata, coord_type="generic", n_neighs=6, key_added="try")
elif GRAPH == "radius":
    sq.gr.spatial_neighbors(adata, coord_type="generic", radius=30.0, key_added="try")
else:
    raise ValueError('GRAPH must be "delaunay", "knn" or "radius"')

deg = np.asarray(adata.obsp["try_connectivities"].sum(axis=1)).ravel()
print(f"{GRAPH}: mean {deg.mean():.1f} neighbours per cell, "
      f"range {deg.min():.0f} to {deg.max():.0f}")
if deg.max() - deg.min() > 20:
    print("Note the huge range — this graph gives dense regions far more neighbours,")
    print("which will confound any per-cell statistic with local cell density.")

In [ ]:
# Long edges are cells "touching" across empty space — a lumen, a tear, the section
# edge. Trim them, or they invent contacts that do not exist in the tissue.
CUT = 40  # um; inspect the histogram before choosing
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(d, bins=80, color=NAVY)
ax.axvline(CUT, color=GOLD, lw=2, label=f"cut at {CUT} um")
ax.set_xlabel("edge length (um)"); ax.set_ylabel("edges"); ax.set_xlim(0, 200)
ax.legend(); sns.despine(); plt.show()
print(f"{(d > CUT).mean():.1%} of edges would be removed")

In [ ]:
import scipy.sparse as sp

dist = adata.obsp["spatial_distances"].tocoo()
keep = dist.data <= CUT
trimmed = sp.coo_matrix((np.ones(keep.sum()), (dist.row[keep], dist.col[keep])),
                        shape=dist.shape).tocsr()
adata.obsp["spatial_connectivities"] = trimmed
print(f"mean neighbours after trimming: {np.asarray(trimmed.sum(axis=1)).ravel().mean():.1f}")

In [ ]:
# Draw the graph on a small patch — this is the object every statistic below
# runs on. Put it on the DAPI channel so you can see the tissue it describes.
PATCH = 150.0     # um; 100-200 is readable, beyond that it turns into spaghetti

x, y = adata.obsm["spatial"].T          # also set in the setup cell; repeated so
                                        # this cell can be re-run on its own

if img_ext is not None:
    # centre the patch inside the imaged window, otherwise there is no image
    # underneath it
    cx = (img_ext[0] + img_ext[1]) / 2
    cy = (img_ext[2] + img_ext[3]) / 2
else:
    cx, cy = x.mean(), y.mean()
x0, y0 = cx - PATCH / 2, cy - PATCH / 2

m = (x >= x0) & (x < x0 + PATCH) & (y >= y0) & (y < y0 + PATCH)
idx = np.where(m)[0]
print(f"{len(idx)} cells in a {PATCH:.0f} x {PATCH:.0f} um patch at ({x0:.0f}, {y0:.0f})")
if len(idx) == 0:
    raise RuntimeError("no cells in the patch — the image window and the cell crop "
                       "may not overlap; check data/crop_metadata.json")

sub = trimmed[idx][:, idx].tocoo()
cats = adata.obs["cell_type"].astype("category")
cm = plt.get_cmap("tab20")

fig, axes = plt.subplots(1, 2, figsize=(15, 7.4))
for ax, with_image in zip(axes, [True, False]):
    if with_image and img is not None:
        show_morphology(ax, img, img_ext, channel=0)
        edge, halo = "#FFD166", "white"
    else:
        edge, halo = "0.75", "white"

    for i, j in zip(sub.row, sub.col):
        if i < j:
            ax.plot([x[idx[i]], x[idx[j]]], [y[idx[i]], y[idx[j]]],
                    lw=0.8, color=edge, alpha=0.9, zorder=2)
    ax.scatter(x[idx], y[idx], s=34, zorder=3, linewidths=0.5, edgecolors=halo,
               c=[cm(c % 20) for c in cats.cat.codes[idx]])

    ax.set_xlim(x0, x0 + PATCH)
    ax.set_ylim(y0 + PATCH, y0)        # descending: image convention
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title("graph on DAPI" if with_image and img is not None
                 else "graph alone")
plt.tight_layout(); plt.show()

### Look at the left panel properly

This is the moment to check that your graph describes the tissue rather than a
point cloud you happen to have coordinates for.

- **Do the edges cross visible boundaries they should not?** An edge spanning a
  lumen, a vessel wall or a capsule is a "contact" between cells that never touch.
  That is what the length trim above was for — and you can now see whether it worked.
- **Do nuclei without a coloured dot appear?** Those are cells that failed QC in
  notebook 02 and were removed. Notice they are not randomly placed.
- **Does one dot sit over two nuclei?** A segmentation merge, visible directly.
- **Are there dots with no nucleus under them?** A polygon over extracellular space.

Every neighbourhood statistic for the rest of the day inherits whatever you see here.
If the graph looks wrong on the image, fix it now rather than interpreting the
z-scores it produces.

> The right panel is the same graph without the image, which is what you would
> normally plot. Compare the two. The version without morphology looks perfectly
> convincing — that is the problem with it.

## 2. Neighbourhood enrichment — who sits next to whom?

For every pair of cell types, count edges between them, then compare with a null
made by shuffling the cell-type labels while keeping the graph fixed. The z-score is
"more/fewer contacts than expected by chance given the composition".

In [ ]:
sq.gr.nhood_enrichment(adata, cluster_key="cell_type", n_perms=1000, seed=0)
sq.pl.nhood_enrichment(adata, cluster_key="cell_type", method="average",
                       cmap="RdBu_r", vmin=-60, vmax=60, figsize=(7, 6))

### How to read this, and how not to

- Red on the diagonal = that cell type is **clustered** (tumour nests, lymphoid
  aggregates). Almost every cell type shows this; it is not a finding on its own.
- Red off-diagonal = two types are **found together more than chance**.
- Blue off-diagonal = **segregated**. This is often the more interesting direction —
  immune exclusion from tumour nests is a blue square, and it is a real, published,
  clinically relevant phenotype.

**Three traps.**

1. The null shuffles labels globally. In a section with large-scale structure,
   *everything* looks enriched with everything nearby. Interpret relative to the
   overall pattern, not against zero.
2. z-scores scale with the number of cells. A z of 200 in 50,000 cells is not
   ten times more meaningful than a z of 20 in 5,000. Compare effect sizes across
   datasets, never raw z-scores.
3. One section, one patient. **n = 1.** Anything here is a hypothesis.

## 3. At what distance? Co-occurrence and Ripley

Neighbourhood enrichment answers "adjacent or not" at one scale. Real biology has a
scale: juxtacrine signalling acts over ~10 um, a chemokine gradient over ~100 um,
tissue compartments over millimetres. Co-occurrence measures the enrichment as a
*function of radius*.

In [ ]:
sq.gr.co_occurrence(adata, cluster_key="cell_type", n_splits=None)

types = list(adata.obs["cell_type"].cat.categories)
anchor = next((t for t in types if "Epithelial" in t or "umour" in t), types[0])
sq.pl.co_occurrence(adata, cluster_key="cell_type", clusters=anchor, figsize=(7, 4.5))
print(f"anchor cell type: {anchor}")

The y-axis is p(other | anchor within r) / p(other), i.e. enrichment relative to
random. Above 1 = attracted, below 1 = repelled, and **where the curve crosses 1
is the length scale of the interaction**. That number — in microns — is the kind of
result you can put in a figure legend and compare between patients.

In [ ]:
# Ripley's L: is a single cell type more clustered than random, and at what radius?
target = next((t for t in types if "T cell" in t), types[min(1, len(types) - 1)])
sq.gr.ripley(adata, cluster_key="cell_type", mode="L", n_simulations=100, seed=0)
sq.pl.ripley(adata, cluster_key="cell_type", mode="L", figsize=(7, 4.5))
print(f"look at: {target}")

Above the grey envelope = clustered; below = dispersed/regular. Regular spacing is
rare and interesting — it appears for things under active spatial control, like
tissue-resident macrophages tiling a territory.

### Exercise 4.1
Pick the immune population in this section. Is it clustered, dispersed, or random?
At which radius is the deviation largest, and what tissue structure has that size?

In [ ]:
# your code here

## 4. Spatially variable genes — no clusters required

Everything above needs cell-type labels, which means it inherits every mistake from
notebook 03. **Moran's I** does not: it asks directly whether a gene's expression is
autocorrelated in space. Neighbouring cells more similar than distant ones → high I.

This is a genuinely different question from "is this gene a marker". A gene can be
strongly cell-type-specific and *not* spatially structured (if that cell type is
scattered), and it can be spatially structured without marking any cluster (a
gradient across a compartment).

In [ ]:
sq.gr.spatial_autocorr(adata, mode="moran", n_perms=100, n_jobs=1, seed=0)
mi = adata.uns["moranI"]
print("most spatially structured genes:")
display(mi.head(15).round(4))
print("\nleast (essentially random in space):")
display(mi.tail(5).round(4))

In [ ]:
top = mi.head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
X = adata[:, top].X
X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
for k, (g, ax) in enumerate(zip(top, axes.ravel())):
    v = X[:, k]
    order = np.argsort(v)            # draw high-expressing cells on top
    pc = ax.scatter(x[order], y[order], c=v[order], s=1.1, cmap="magma",
                    vmax=np.percentile(v, 99), linewidths=0, rasterized=True)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{g}   Moran's I = {mi.loc[g, 'I']:.2f}")
    plt.colorbar(pc, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

> **Tie back to notebook 02.** Moran's I is an independent check on the detection
> threshold you set there. A gene just below the count cutoff but with clear spatial
> autocorrelation is almost certainly real — noise does not form patches. Cross the
> two: `adata.var["above_background"]` against the `moranI` table.

### Exercise 4.2
Find a gene with **high Moran's I but low variance between your cell-type clusters**
(compare `mi` against `sc.tl.rank_genes_groups` output). What could produce that?
Hint: think about gradients, and about things that are not cell types.

In [ ]:
# your code here

## 5. Niches — the unit of tissue organisation

A cell type is a property of a cell. A **niche** is a property of a *neighbourhood*:
the recurring combinations of cell types that appear together across the section.
Tumour nest core, invasive front, vascular stroma, tertiary lymphoid structure —
these are niches, and they are the level at which most tissue biology happens.

The standard recipe is simple enough to write out in full, and worth writing out so
you know what the packages are doing:

1. for each cell, the composition of its neighbourhood (fraction of each cell type)
2. cluster those composition vectors
3. each cluster is a niche

In [ ]:
# Step 1: neighbourhood composition
onehot = pd.get_dummies(adata.obs["cell_type"]).to_numpy().astype(float)
A = adata.obsp["spatial_connectivities"]
A = A + sp.eye(A.shape[0], format="csr")          # include the cell itself
comp = A @ onehot
comp = comp / comp.sum(axis=1, keepdims=True)

comp_df = pd.DataFrame(comp, index=adata.obs_names,
                       columns=adata.obs["cell_type"].cat.categories)
comp_df.head().round(3)

In [ ]:
# Step 2: cluster the composition vectors
from sklearn.cluster import KMeans

K = 6                      # try 4-10; there is no correct answer, only a useful one
km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(comp)
adata.obs["niche"] = pd.Categorical([f"N{i}" for i in km.labels_])
adata.obsm["nbhd_composition"] = comp
print(adata.obs["niche"].value_counts().sort_index())

> **Try it yourself — how many niches?**
>
> Change `TRY_K` and look at the map. Too few and everything merges into
> "tumour" and "not tumour"; too many and you get regions you cannot name.
> The test is not a statistic: can you point at each niche and say what it is?

In [ ]:
TRY_K = 4        # <-- CHANGE THIS (try 3, 4, 6, 8, 10)

from sklearn.cluster import KMeans
labels = KMeans(n_clusters=TRY_K, n_init=10, random_state=0).fit_predict(comp)

fig, ax = plt.subplots(figsize=(6.5, 6.5))
cmn = plt.get_cmap("Set2")
for k in range(TRY_K):
    m = labels == k
    ax.scatter(x[m], y[m], s=1.4, color=cmn(k % 8), label=f"N{k} ({m.sum():,})",
               linewidths=0, rasterized=True)
ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
ax.legend(markerscale=8, loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)
ax.set_title(f"K = {TRY_K}")
plt.tight_layout(); plt.show()

In [ ]:
# Step 3: what is each niche made of?
profile = comp_df.groupby(adata.obs["niche"].values, observed=True).mean()
fig, ax = plt.subplots(figsize=(9, 4.2))
sns.heatmap(profile, cmap="Blues", annot=True, fmt=".2f", ax=ax,
            cbar_kws={"label": "mean neighbourhood fraction"})
ax.set_ylabel("niche"); plt.xticks(rotation=40, ha="right")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
cmn = plt.get_cmap("Set2")
for i, name in enumerate(adata.obs["niche"].cat.categories):
    m = (adata.obs["niche"] == name).to_numpy()
    axes[0].scatter(x[m], y[m], s=1.4, color=cmn(i % 8), label=name, linewidths=0, rasterized=True)
axes[0].legend(markerscale=8, loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)
axes[0].set_title("niches")

cmt = plt.get_cmap("tab20")
for i, name in enumerate(adata.obs["cell_type"].cat.categories):
    m = (adata.obs["cell_type"] == name).to_numpy()
    axes[1].scatter(x[m], y[m], s=1.4, color=cmt(i % 20), linewidths=0, rasterized=True)
axes[1].set_title("cell types, same section")
for ax in axes:
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### The comparison is the point

The niche map is **smoother and more anatomical** than the cell-type map. It looks
more like an H&E than a scatter plot does, because it is describing tissue
compartments rather than individual cells.

Now name them. Go through your niches and give each an anatomical label — "tumour
nest core", "stromal band", "perivascular", "immune infiltrate at the invasive
front". This is where a room full of anatomists has an advantage over a room full of
bioinformaticians: you already know what these structures are called and what they
mean.

### Exercise 4.3
Re-run with `K = 4` and `K = 10`. What appears and what merges? Then justify your
chosen K — not by a silhouette score, but by whether the niches correspond to
structures you can name.

### Exercise 4.4
Does any **gene** differ between two niches *within the same cell type*? That is,
take one cell type, split it by niche, and run differential expression. A tumour cell
at the invasive front versus one in the nest core — same cell type, different
neighbourhood, possibly different state. This design is impossible in scRNA-seq.

In [ ]:
# your code here — sc.tl.rank_genes_groups on a subset, groupby="niche"

In [ ]:
adata.write_h5ad(DATA / "ovarian_niches.h5ad", compression="gzip")
print("saved -> ovarian_niches.h5ad")

---
**Next:** `05_beyond_single_cell.ipynb` — distance, contact, and the sub-cellular layer.